In [ ]:
# ============================================================
# 22. Logistic Results (Robust Inference)
# ============================================================
#
# 목적:
# 1. Robust SE 기반 계수 추정
# 2. Odds Ratio 및 신뢰구간 산출
# 3. SHAP 결과와 비교 가능한 해석 테이블 생성
#
# 입력:
# - 학습된 로지스틱 모델
#
# 출력:
# - logistic_or_ci_summary.csv
#
# 활용:
# - 보고서
# - 대시보드 설명 레이어
# ============================================================


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

In [3]:
INPUT_PATH = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/feature_table_logistic_base.csv"
OUTPUT_PATH = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/reports/logistic_or_ci_summary.csv"
LABEL_COL = "death_next_24h"

In [5]:
df = pd.read_csv(INPUT_PATH)

In [6]:
X = df.drop(columns=[LABEL_COL, "stay_id"])
y = df[LABEL_COL]
groups = df["stay_id"]

In [7]:
binary_cols = [c for c in X.columns if X[c].dropna().isin([0, 1]).all()]
continuous_cols = [c for c in X.columns if c not in binary_cols]

In [8]:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, _ = next(gss.split(X, y, groups))
X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

In [9]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])

In [17]:
# 분산이 0이거나 거의 0인 컬럼 제거
var = X_train_scaled.var()
low_var_cols = var[var < 1e-6].index.tolist()

print("Low variance cols:", low_var_cols)

X_train_scaled = X_train_scaled.drop(columns=low_var_cols)

Low variance cols: []


In [11]:
# continuous_cols 기준으로만
X_cont = X_train_scaled[continuous_cols]
print(X_cont.shape)

(752283, 61)
